# Movie identification with BERG features + trained Ridge encoder (Option B)

In **Option A** (`03_berg_in_silico_predict.ipynb`) we used BERG's pretrained
in silico EEG *directly* and matched it against the real EEG. With zero tuning
it only reached ~2x chance (top-1 0.0196, top-5 0.0784) because the BERG model
was trained on people viewing **still images** (THINGS-EEG2), not **movies**,
and in a different channel/time layout than this dataset.

**Option B** keeps BERG as the *feature extractor* but learns a subject-specific
mapping, exactly like the AlexNet pipeline in notebooks 01/02:

```
video -> frames -> BERG in silico EEG (per frame, averaged)  =  X  (features)
X -> StandardScaler -> PCA(0.95) -> StandardScaler -> RidgeCV -> real EEG
```

We train the Ridge encoder on this subject's **real training EEG** so the model
learns how this person's brain responses relate to BERG's representation. Then
we predict EEG for the test videos and match against the real test EEG.

Why this should beat Option A: instead of trusting BERG's absolute response, we
only use it as a rich stimulus descriptor and let RidgeCV fit the subject- and
task-specific relationship, the same thing that made notebook 02 work.

## Requirements (same as notebook 03)
- Python >= 3.11, `berg` installed, and the subject-01 `vit_b_32` EEG weights
  downloaded under `BERG_DIR` (already done for notebook 03).

In [13]:
# --- Windows UTF-8 fix + clean berg import (see notebook 03 for why) ---
import builtins, sys
if not getattr(builtins, '_utf8_open_patched', False):
    _orig_open = builtins.open
    def _utf8_open(*a, **kw):
        mode = kw.get('mode', a[1] if len(a) > 1 else 'r')
        if 'b' not in mode and kw.get('encoding') is None:
            kw['encoding'] = 'utf-8'
        return _orig_open(*a, **kw)
    builtins.open = _utf8_open
    builtins._utf8_open_patched = True
for _m in [m for m in list(sys.modules) if m == 'berg' or m.startswith('berg.')]:
    del sys.modules[_m]

import os, glob
import numpy as np
import cv2
from scipy.signal import decimate
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

device: cuda


In [14]:
# ---------------- Config ----------------
SUB        = 2

BERG_DIR   = './brain-encoding-response-generator'
MODEL_ID   = 'eeg-things_eeg_2-vit_b_32'

STIM_DIR        = './stimuli_train'
STIM_TEST_DIR   = './stimuli_test'
FRAME_DIR       = './frames'
FRAME_TEST_DIR  = './frames_test'
TRAIN_DIR       = './trn'
TEST_DIR        = './tst'
GT_DIR          = './tst_stim'

# Cache BERG features so the slow extraction runs only once
CACHE_DIR       = './features_berg'
os.makedirs(CACHE_DIR, exist_ok=True)

SFREQ      = 500
DEC        = 5
N_FRAMES   = 8
IMG_SIZE   = 224

tst_ids = list(range(1001, 1103))
print(len(tst_ids), 'test ids')

102 test ids


## 1. Load the real EEG (train + test) and decimate (same as notebooks 01/02)

In [15]:
# Training EEG: npz keyed by stimulus id
trn = np.load(f'{TRAIN_DIR}/sub-{SUB:02d}_trn.npz')
trn_ids = np.array(sorted(trn.files, key=int), dtype=int)
trn_eeg = np.stack([trn[str(i)] for i in trn_ids])        # (968, 17, 1500)
trn_eeg = decimate(trn_eeg, DEC, axis=-1, ftype='fir', zero_phase=True)
print('train EEG:', trn_eeg.shape, 'ids:', trn_ids[:3], '...', trn_ids[-2:])

# Test EEG
tst_eeg = np.load(f'{TEST_DIR}/sub-{SUB:02d}_tst.npy')     # (102, 17, 1500)
tst_eeg = decimate(tst_eeg, DEC, axis=-1, ftype='fir', zero_phase=True)
n_stim, n_chan, n_time = tst_eeg.shape
print('test EEG:', tst_eeg.shape)

train EEG: (972, 17, 300) ids: [1 2 3] ... [ 999 1000]
test EEG: (102, 17, 300)


## 2. Frame helpers (Windows-safe) + make sure frames exist
Training frames already live in `frames/`; test frames in `frames_test/`.

In [16]:
def video_for(stim_id, vids):
    target = f'{int(stim_id)}.mp4'
    hits = [v for v in vids if os.path.basename(v) == target]
    assert hits, f'no video found for stimulus {stim_id} (expected {target})'
    return hits[0]

def dump_frames(stim_ids, stim_dir, out_dir, n_frames=N_FRAMES):
    vids = sorted(glob.glob(os.path.join(stim_dir, '*.mp4')))
    os.makedirs(out_dir, exist_ok=True)
    for sid in stim_ids:
        done = sorted(glob.glob(os.path.join(out_dir, f'{int(sid):05d}_f*.jpg')))
        if len(done) >= n_frames:
            continue
        cap = cv2.VideoCapture(video_for(sid, vids))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        picks = np.linspace(0, max(total - 1, 0), n_frames).astype(int)
        for k, fno in enumerate(picks):
            path = os.path.join(out_dir, f'{int(sid):05d}_f{k:02d}.jpg')
            if os.path.exists(path):
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fno))
            ok, frame = cap.read()
            if ok:
                cv2.imwrite(path, frame)
        cap.release()

def frames_for(sid, out_dir):
    return sorted(glob.glob(os.path.join(out_dir, f'{int(sid):05d}_f*.jpg')))

def load_rgb(path):
    return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)

# Ensure frames are present (skips work if already dumped)
dump_frames(trn_ids, STIM_DIR, FRAME_DIR)
dump_frames(tst_ids, STIM_TEST_DIR, FRAME_TEST_DIR)
print('train frames:', len(glob.glob(f'{FRAME_DIR}/*.jpg')),
      '| test frames:', len(glob.glob(f'{FRAME_TEST_DIR}/*.jpg')))

train frames: 8000 | test frames: 816


## 3. Load BERG and define the feature extractor
Same verified encode path as notebook 03: square, channels-first uint8 input;
output `(batch, 4 reps, 63, 140)` averaged over the repetition axis.

In [17]:
from berg import BERG
berg = BERG(berg_dir=BERG_DIR)
eeg_model = berg.get_encoding_model(MODEL_ID, subject=1, device=DEVICE)

def to_berg_stimulus(rgb_images):
    out = []
    for im in rgb_images:
        sq = cv2.resize(im, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        out.append(np.transpose(sq, (2, 0, 1)))      # HWC -> CHW
    return np.stack(out).astype('uint8')

def berg_response(rgb_images):
    """list of RGB frames -> (n, 63, 140) in silico responses (reps averaged)."""
    stim = to_berg_stimulus(rgb_images)
    out = np.asarray(berg.encode(eeg_model, stim))
    if out.ndim == 4:
        out = out.mean(axis=1)
    return out

Model loaded on cuda for subject 1


## 4. Build BERG feature matrices for train and test videos
For each video we encode its 8 frames and average into one feature vector.
Results are cached to `features_berg/` so this slow step runs only once.

In [18]:
def build_features(ids, frame_dir, cache_name):
    cache_path = os.path.join(CACHE_DIR, cache_name)
    if os.path.exists(cache_path):
        data = np.load(cache_path)
        if np.array_equal(data['ids'], np.asarray(ids)):
            print('loaded cached', cache_name, data['X'].shape)
            return data['X']
    rows = []
    from time import time
    t0 = time()
    for n, sid in enumerate(ids):
        paths = frames_for(sid, frame_dir)
        assert paths, f'no frames for stimulus {sid} in {frame_dir}'
        resp = berg_response([load_rgb(p) for p in paths])   # (n_frames, 63, 140)
        rows.append(resp.reshape(resp.shape[0], -1).mean(axis=0))
        if (n + 1) % 50 == 0 or n + 1 == len(ids):
            print(f'  {n+1}/{len(ids)} videos  ({time()-t0:.0f}s)')
    X = np.stack(rows)
    np.savez_compressed(cache_path, X=X, ids=np.asarray(ids))
    print('saved', cache_name, X.shape)
    return X

X_trn = build_features(trn_ids, FRAME_DIR, f'berg_trn_sub-{SUB:02d}.npz')
X_tst = build_features(tst_ids, FRAME_TEST_DIR, f'berg_tst_sub-{SUB:02d}.npz')
print('X_trn', X_trn.shape, '| X_tst', X_tst.shape)

Encoding EEG responses: 100%|██████████| 1/1 [00:00<00:00,  2.71it/s, Encoded images=8, Total images=8]


  50/972 videos  (45s)


Encoding EEG responses: 100%|██████████| 1/1 [00:00<00:00,  2.76it/s, Encoded images=8, Total images=8]


  100/972 videos  (74s)


Encoding EEG responses: 100%|██████████| 1/1 [00:00<00:00,  2.66it/s, Encoded images=8, Total images=8]


...

  50/102 videos  (32s)


Encoding EEG responses: 100%|██████████| 1/1 [00:00<00:00,  1.14it/s, Encoded images=8, Total images=8]


  100/102 videos  (71s)


Encoding EEG responses: 100%|██████████| 1/1 [00:00<00:00,  1.92it/s, Encoded images=8, Total images=8]


  102/102 videos  (73s)
saved berg_tst_sub-02.npz (102, 8820)
X_trn (972, 8820) | X_tst (102, 8820)


## 5. Train the encoder: BERG features -> real EEG (same recipe as notebook 01)

In [19]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV

sc  = StandardScaler().fit(X_trn)
X_trn_z = sc.transform(X_trn)

pca = PCA(n_components=0.95, svd_solver='full').fit(X_trn_z)
X_trn_p = pca.transform(X_trn_z)

sc2 = StandardScaler().fit(X_trn_p)
X_trn_s = sc2.transform(X_trn_p)
print('PCA components:', X_trn_p.shape[1])

model = RidgeCV(alphas=np.logspace(-3, 6, 20))
model.fit(X_trn_s, trn_eeg.reshape(len(trn_ids), -1))
print('alpha =', model.alpha_)

PCA components: 146
alpha = 4281.332398719387


## 6. Predict EEG for the test videos, then match against the real test EEG

In [20]:
# Map test BERG features through the SAME transforms (no re-fitting)
X_tst_z = sc.transform(X_tst)
X_tst_p = pca.transform(X_tst_z)
X_tst_s = sc2.transform(X_tst_p)

pred_tst_eeg = model.predict(X_tst_s).reshape(len(tst_ids), n_chan, n_time)
print('predicted test EEG:', pred_tst_eeg.shape)

predicted test EEG: (102, 17, 300)


In [21]:
# Ground truth: true video id per measured test recording
gt = np.load(f'{GT_DIR}/sub-{SUB:02d}_tst_stim.npy')
order = np.argsort(gt)                      # align measured rows to sorted order

# Same informative window + correlation matching as notebook 02
M = tst_eeg[order][:, :, 5:60].reshape(len(tst_ids), -1)       # measured (sorted)
P = pred_tst_eeg[:, :, 5:60].reshape(len(tst_ids), -1)         # predicted (video order)

def zrows(A):
    return (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)

Mz, Pz = zrows(M), zrows(P)
S = Mz @ Pz.T / M.shape[1]                   # (102 measured, 102 predicted videos)
print('similarity matrix:', S.shape)

similarity matrix: (102, 102)


In [22]:
video_ids = np.array(tst_ids)                # column j -> video 1001+j
order_idx = np.argsort(-S, axis=1)
ranked    = video_ids[order_idx]             # (102, 102) best-first video ids

truth = gt[order]                            # true video id for each measured row
top1 = (ranked[:, 0] == truth).mean()
top5 = (ranked[:, :5] == truth[:, None]).any(axis=1).mean()
ranks = (-S).argsort(1).argsort(1)[np.arange(len(tst_ids)), np.arange(len(tst_ids))]

print(f'Option B (BERG features + Ridge)')
print(f'  top-1  {top1:.4f}   (chance {1/len(tst_ids):.4f})')
print(f'  top-5  {top5:.4f}   (chance {5/len(tst_ids):.4f})')
print(f'  median rank {np.median(ranks)+1:.1f}   (chance ~{(len(tst_ids)+1)/2:.1f})')
print('\nFor comparison:')
print('  Option A (BERG direct):  top-1 0.0196,  top-5 0.0784')
print('  AlexNet (notebook 02):   top-1 0.0392,  top-5 0.1176')

Option B (BERG features + Ridge)
  top-1  0.0588   (chance 0.0098)
  top-5  0.2157   (chance 0.0490)
  median rank 21.0   (chance ~51.5)

For comparison:
  Option A (BERG direct):  top-1 0.0196,  top-5 0.0784
  AlexNet (notebook 02):   top-1 0.0392,  top-5 0.1176


In [23]:
ranked

array([[1101, 1023, 1062, ..., 1027, 1048, 1015],
       [1027, 1076, 1034, ..., 1029, 1101, 1031],
       [1032, 1078, 1054, ..., 1014, 1073, 1074],
       ...,
       [1100, 1075, 1079, ..., 1031, 1026, 1060],
       [1031, 1054, 1101, ..., 1100, 1057, 1063],
       [1039, 1079, 1068, ..., 1101, 1026, 1054]], shape=(102, 102))

In [24]:
np.save(f'sub-{SUB:02d}_ranked_candidate_list_berg_ridge.npy', ranked)
print('saved sub-{:02d}_ranked_candidate_list_berg_ridge.npy'.format(SUB))

saved sub-02_ranked_candidate_list_berg_ridge.npy


## Notes
- Here BERG's `(63, 140)` in silico response is just a **stimulus descriptor**;
  RidgeCV learns the subject-specific map from it to the real `(17, 300)` EEG,
  so the channel/time mismatch that hurt Option A no longer matters.
- If this still trails the AlexNet baseline, BERG's image-trained representation
  may simply carry less movie-relevant signal than AlexNet's conv features for
  this task. Worth trying: use an intermediate BERG/ViT feature instead of the
  final in silico EEG, or concatenate BERG + AlexNet features.
- To run other subjects, change `SUB` (training EEG exists for 01-06; ground
  truth for scoring only exists for 01-04).